# House Price Prediction

Predicting house prices using the house data from Kaggle.
Dataset: https://www.kaggle.com/datasets/shree1992/housedata


## 1. Load the data

I use kagglehub so the dataset gets downloaded automatically, no manual download needed.


In [ ]:
!pip install kagglehub -q

import kagglehub, os
path = kagglehub.dataset_download('shree1992/housedata')
print(os.listdir(path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(os.path.join(path, 'data.csv'))
df.head()


## 2. Quick look at the data

Checking the shape, dtypes and missing values first.


In [ ]:
print(df.shape)
df.info()
df.isnull().sum()[df.isnull().sum() > 0]


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['price'], kde=True)
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(x='sqft_living', y='price', data=df)
plt.show()

plt.figure(figsize=(14,8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.show()


## 3. Cleaning

The sqft_above column has commas in the numbers so it loaded as object, fixing that.
Dropped columns that don't help (date, street, country).
Also removing extreme prices using the 1% and 99% quantiles to get rid of outliers.


In [ ]:
df = df.dropna(subset=['price'])

if df['sqft_above'].dtype == object:
    df['sqft_above'] = df['sqft_above'].str.replace(',', '').astype(float)

df = df.drop(columns=['date', 'street', 'country'], errors='ignore')

from sklearn.preprocessing import LabelEncoder
df['city_encoded'] = LabelEncoder().fit_transform(df['city'].astype('category'))
df['statezip_encoded'] = LabelEncoder().fit_transform(df['statezip'].astype('category'))
df = df.drop(columns=['city', 'statezip'])

low, high = df['price'].quantile([0.01, 0.99])
df = df[(df['price'] >= low) & (df['price'] <= high)]
df.shape


## 4. Training

Split the data, scale it, then try 3 models and see which one does best.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42)
}

scores = {}
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name] = model
    scores[name] = {
        'R2': round(r2_score(y_test, pred), 4),
        'MAE': round(mean_absolute_error(y_test, pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2)
    }

pd.DataFrame(scores).T


## 5. Best model and saving

Picking the model with the highest R2, plotting actual vs predicted, then saving everything with joblib.


In [ ]:
best_name = pd.DataFrame(scores).T['R2'].idxmax()
best_model = fitted[best_name]
print('best model:', best_name)

pred = best_model.predict(X_test)
plt.figure(figsize=(7,7))
sns.scatterplot(x=y_test, y=pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('actual')
plt.ylabel('predicted')
plt.show()


In [ ]:
os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(best_model, 'model_artifacts/house_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(X.columns.tolist(), 'model_artifacts/features.pkl')
os.listdir('model_artifacts')
